# Parallel Processing with Dask

`pygeodata` integrates with [Dask](https://dask.org/) via `build_dask_graph`.
This constructs a lazy computation graph that respects data dependencies,
allowing Dask to parallelise work across cores or a distributed cluster.

This notebook covers:

- Processing multiple specs in parallel
- Nested DAGs: wiring upstream dependencies automatically
- Using a distributed cluster

In [ ]:
import os, sys
os.chdir('../../..')
sys.path.insert(0, 'docs/loaders')

In [ ]:
from pathlib import Path

from affine import Affine
from pyproj import CRS

from pygeodata import SpatialSpec, get_config
from pygeodata.parallel import build_dask_graph

get_config().update(path_cache=Path('data/processed'))

## 1. Process one loader at multiple resolutions in parallel

A common pattern is to produce the same output at several resolutions or
extents. `build_dask_graph` wraps each call as a lazy Dask delayed node;
`dask.compute(*tasks)` runs them in parallel.

Here we reproject the water-table depth raster to three different resolutions
over Australia.

In [ ]:
import dask
from pipeline import WaterTableDepthLoader

loader = WaterTableDepthLoader()

# Three resolutions over Australia in EPSG:4326
resolutions = [0.05, 0.1, 0.25]  # degrees per pixel
specs = [
    SpatialSpec(
        crs=CRS.from_epsg(4326),
        transform=Affine(res, 0, 110, 0, -res, -9),
        shape=(int(36 / res), int(45 / res)),
    )
    for res in resolutions
]

tasks = [build_dask_graph(loader, spec=s) for s in specs]
print('Task keys:')
for t, res in zip(tasks, resolutions):
    print(f'  {res}°  →  {t.key}')

# Run all three in parallel using the local threaded scheduler
dask.compute(*tasks, scheduler='threads', num_workers=3)
print('Done.')

## 2. Nested DAG: dependencies wired automatically

Pass the root loader — `build_dask_graph` recursively discovers all embedded
upstream `Data` parameters and wires them as Dask dependencies in the correct
order. You don't need to schedule upstream tasks separately.

In [ ]:
from pipeline import LandWaterTableDepth, WaterTableDepthLoader, CountryMaskLoader

spec = SpatialSpec.from_raster_file('data/wtd.tif')

land_wtd = LandWaterTableDepth(
    wtd=WaterTableDepthLoader(),
    mask=CountryMaskLoader(),
)

# build_dask_graph discovers WaterTableDepthLoader and CountryMaskLoader
# as upstream dependencies and wires them into the graph automatically
task = build_dask_graph(land_wtd, spec=spec)
print('Root task key:', task.key)

dask.compute(task, scheduler='synchronous')
print('Pipeline complete.')

## 3. Distributed cluster

Connect a Dask `Client` before calling `compute()` to distribute work across
many workers. The API is identical — only the scheduler changes.

```python
from dask.distributed import Client

client = Client(n_workers=8, threads_per_worker=1)
print('Dashboard:', client.dashboard_link)

dask.compute(*tasks)   # Dask picks up the distributed client automatically

client.close()
```